# Experimenting with step size for a galaxy bigger than the localization

In [26]:
# imports
from importlib import reload
import os
from importlib.resources import files as resource_files

import numpy as np

import pandas

from astropy.coordinates import SkyCoord
from astropy.coordinates import offset_by
from astropy import units
#from astropy.io import fits

from astropath import path
from astropath import localization
from astropath import bayesian

# Convenience class

In [4]:
Path = path.PATH()

# Faux FRB

In [2]:
## FRB Coord
frb_coord = SkyCoord('21h44m25.255s -40d54m00.10s', frame='icrs')
frb_coord

<SkyCoord (ICRS): (ra, dec) in deg
    (326.10522917, -40.90002778)>

# Prior

In [25]:
theta_prior = dict(max=6., PDF='exp', scale=1.)

# Localization

In [49]:
# Small, but not tiny
eellipse = dict(a=0.5, b=0.5, theta=0.)
# Tiny
tiny_eellipse = dict(a=0.1, b=0.1, theta=0.)

## Build it

In [29]:
Path.init_localization('eellipse', center_coord=frb_coord, eellipse=eellipse)

In [30]:
Path.localiz

{'type': 'eellipse',
 'center_coord': <SkyCoord (ICRS): (ra, dec) in deg
     (326.10522917, -40.90002778)>,
 'eellipse': {'a': 0.5, 'b': 0.5, 'theta': 0.0}}

# Galaxy

## Position

In [17]:
# 0.5" North
gal_coord = frb_coord.directional_offset_by(0.*units.deg, 0.5*units.arcsec)
gal_coord, gal_coord.separation(frb_coord).to('arcsec')

(<SkyCoord (ICRS): (ra, dec) in deg
     (326.10522917, -40.89988889)>,
 <Angle 0.5 arcsec>)

## Size

In [34]:
# Medium (5")
gal_size = np.array([5.]) # arcsec
box_hwidth = 50.

# Calculate

## Tiny step_size

In [39]:
reload(bayesian)
step_size = 0.05
L_wx, p_wOi, grid_p, p_xOis = bayesian.px_Oi_fixedgrid(box_hwidth, Path.localiz, np.array([gal_coord]),
                    gal_size, theta_prior, step_size=step_size, return_debug=True)

In [45]:
print(f'L_wx: {np.sum(L_wx) * step_size**2}')
print(f'p_wO: {np.sum(p_wOi) * step_size**2}')
print(f'p_xO: {p_xOis}')

L_wx: 0.9990002500065099
p_wO: 0.9989995728142178
p_xO: 0.005565669003159662


## Vary it

In [60]:
Path.init_localization('eellipse', center_coord=frb_coord, eellipse=eellipse)
Path.localiz
for step_size in [0.05, 0.1, 0.25, 0.5, 1., 5.]:
    L_wx, p_wOi, grid_p, p_xOis = bayesian.px_Oi_fixedgrid(box_hwidth, Path.localiz, np.array([gal_coord]),
                    gal_size, theta_prior, step_size=step_size, return_debug=True)
    print('================================================')
    print(f'step: {step_size}')
    print(f'L_wx: {np.sum(L_wx) * step_size**2}')
    print(f'p_wO: {np.sum(p_wOi) * step_size**2}')
    print(f'p_xO_corr: {p_xOis/np.sum(L_wx)/step_size**2}')
    print(f'p_xO: {p_xOis}')

step: 0.05
L_wx: 0.9990002500065099
p_wO: 0.9989995728142178
p_xO_corr: 0.005571238849162845
p_xO: 0.005565669003159662
step: 0.1
L_wx: 0.9980009999805872
p_wO: 0.9980024191155251
p_xO_corr: 0.005576787526152111
p_xO: 0.005565639527779072
step: 0.25
L_wx: 0.9950062499888919
p_wO: 0.9950197864401361
p_xO_corr: 0.005593068199721013
p_xO: 0.005565137815336528
step: 0.5
L_wx: 0.9900249871353524
p_wO: 0.9899999465857889
p_xO_corr: 0.0056171414167946665
p_xO: 0.005561110358899594
step: 1.0
L_wx: 0.9492423957451299
p_wO: 0.9801137413800345
p_xO_corr: 0.005598644562157263
p_xO: 0.005314470777107605
step: 5.0
L_wx: 5.936931936520572e-11
p_wO: 0.8927383033791967
p_xO_corr: 0.003407067919875997
p_xO: 2.0227530343406521e-13


# Tiny localization, 5" galaxy

In [62]:
Path.init_localization('eellipse', center_coord=frb_coord, eellipse=tiny_eellipse)
Path.localiz
for step_size in [0.02, 0.05, 0.1, 0.25, 0.5, 1., 5.]:
    L_wx, p_wOi, grid_p, p_xOis = bayesian.px_Oi_fixedgrid(box_hwidth, Path.localiz, np.array([gal_coord]),
                    gal_size, theta_prior, step_size=step_size, return_debug=True)
    print('================================================')
    print(f'step: {step_size}')
    print(f'L_wx: {np.sum(L_wx) * step_size**2}')
    print(f'p_wO: {np.sum(p_wOi) * step_size**2}')
    print(f'p_xO: {p_xOis}')
    print(f'p_xO_corr: {p_xOis/np.sum(L_wx)/step_size**2}')
    #print(f'p_xO_norm_norm: {p_xOis/np.sum(L_wx)/np.sum(p_wOi)/step_size**4}')

step: 0.02
L_wx: 0.9996000399631986
p_wO: 0.9995974938858885
p_xO: 0.005851396503041362
p_xO_corr: 0.005853737764212962
step: 0.05
L_wx: 0.9990002500317912
p_wO: 0.9989995728142178
p_xO: 0.005851396502948485
p_xO_corr: 0.005857252290739943
step: 0.1
L_wx: 0.9980009889120542
p_wO: 0.9980024191155251
p_xO: 0.00585139640804566
p_xO_corr: 0.00586311684362599
step: 0.25
L_wx: 0.8306016683369597
p_wO: 0.9950197864401361
p_xO: 0.00487868135693966
p_xO_corr: 0.0058736715117702724
step: 0.5
L_wx: 0.028849033861050546
p_wO: 0.9899999465857889
p_xO: 0.0001685103560061478
p_xO_corr: 0.0058411091621912445
step: 1.0
L_wx: 5.32193597973942e-10
p_wO: 0.9801137413800345
p_xO: 2.9945296714021977e-12
p_xO_corr: 0.005626767557524847
step: 5.0
L_wx: 2.7788932288980924e-298
p_wO: 0.8927383033791967
p_xO: 9.465433682953108e-301
p_xO_corr: 0.0034061883286917115
